# Tutorial 04 — Training Utilities

Covers:
1. **Label construction** — `causal_lm_labels` and `make_cdr_infill_batch`
2. **Optimizer & scheduler** — `build_optimizer`, `build_scheduler`
3. **Stage A mini-loop** — projector only, decoder frozen
4. **Stage B mini-loop** — projector + LoRA

No real dataset needed — uses random tensors.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
import torch.nn as nn
from src.utils.tokenizer import AntibodyTokenizer
from src.models.decoder import AntibodyDecoder
from src.models.pooling import build_pooler
from src.models.projectors import build_projector
from src.models.abllava import AbLLaVA

torch.manual_seed(0)
device = 'cpu'
tok = AntibodyTokenizer()

## 1. Label utilities

In [ ]:
from src.training.losses import causal_lm_labels, make_cdr_infill_batch

# causal_lm_labels: shift right, mask prefix (structure tokens) with -100
B, L = 2, 20
prefix_len = 6   # projector tokens
seq_ids = torch.randint(5, tok.vocab_size, (B, L))

labels = causal_lm_labels(seq_ids, prefix_len=prefix_len, pad_id=tok.pad_id)
print("causal_lm_labels:")
print(f"  input shape : {seq_ids.shape}")
print(f"  labels shape: {labels.shape}")
print(f"  first row   : {labels[0].tolist()}")
print(f"  -100 positions (prefix mask + shift): {(labels[0] == -100).sum().item()}")

In [ ]:
# make_cdr_infill_batch: randomly masks one CDR span
cdr_spans = torch.tensor([
    [[26,32],[50,57],[95,102],[143,153],[169,173],[208,216]],
    [[27,33],[51,58],[96,103],[144,154],[170,174],[209,217]],
], dtype=torch.long)

# Use short sequences for demo
seq_ids_short = torch.randint(5, tok.vocab_size, (B, 50))
infill_ids, infill_labels = make_cdr_infill_batch(
    seq_ids_short, cdr_spans, mask_id=tok.mask_id, pad_id=tok.pad_id
)
print("make_cdr_infill_batch:")
print(f"  masked_ids shape : {infill_ids.shape}")
print(f"  labels shape     : {infill_labels.shape}")
mask_count = (infill_ids[0] == tok.mask_id).sum().item()
print(f"  [MASK] tokens in sample 0: {mask_count}")

## 2. Optimizer and LR Scheduler

In [ ]:
from src.training.optim import build_optimizer, build_scheduler

d_model = 128
decoder = AntibodyDecoder(vocab_size=tok.vocab_size, d_model=d_model, n_layers=2, n_heads=4, d_ff=256)
projector = build_projector('mlp', d_in=512, d_out=d_model)
pooler = build_pooler('cdr_mean')
model = AbLLaVA(decoder=decoder, projector=projector, pooler_name='cdr_mean').to(device)

# Stage A: freeze decoder, only projector trains
for p in model.decoder.parameters():
    p.requires_grad_(False)

trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable params (Stage A): {sum(p.numel() for p in trainable):,}")

optimizer = build_optimizer(
    model,
    lr=1e-4,
    weight_decay=0.01,
    projector_lr_multiplier=1.0,
)
scheduler = build_scheduler(
    optimizer,
    warmup_steps=10,
    total_steps=100,
)
print(f"Optimizer param groups: {len(optimizer.param_groups)}")
print(f"Initial LR: {optimizer.param_groups[0]['lr']:.2e}")

## 3. Stage A — 5-step training mini-loop

In [ ]:
B, N = 2, 230

def make_batch():
    return {
        'struct_emb': torch.randn(B, N, 512),
        'cdr_spans': torch.tensor([
            [[26,32],[50,57],[95,102],[143,153],[169,173],[208,216]],
            [[27,33],[51,58],[96,103],[144,154],[170,174],[209,217]],
        ], dtype=torch.long),
        'pad_mask': torch.ones(B, N, dtype=torch.bool),
        'plddt': torch.rand(B, N) * 30 + 70,
        'seq_ids': torch.randint(5, tok.vocab_size, (B, 50)),
        'seq_pad_mask': torch.ones(B, 50, dtype=torch.long),
    }

model.train()
# Re-enable grad on projector
for p in model.projector.parameters():
    p.requires_grad_(True)

losses = []
for step in range(5):
    batch = make_batch()
    optimizer.zero_grad()
    out = model(batch)
    loss = out['loss']
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())
    print(f"  step {step+1}/5  loss={loss.item():.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

print(f"\nLoss trend: {losses[0]:.4f} → {losses[-1]:.4f}")

## 4. Stage B — projector + LoRA on decoder

In [ ]:
from src.models.lora import apply_lora

# Fresh model
decoder2   = AntibodyDecoder(vocab_size=tok.vocab_size, d_model=128, n_layers=2, n_heads=4, d_ff=256)
projector2 = build_projector('mlp', d_in=512, d_out=128)
model2     = AbLLaVA(decoder=decoder2, projector=projector2, pooler_name='cdr_mean')

# Freeze decoder base, apply LoRA
for p in model2.decoder.parameters():
    p.requires_grad_(False)
model2.decoder = apply_lora(model2.decoder, r=4, alpha=16, dropout=0.05, targets=['q_proj','v_proj'])

trainable_b = sum(p.numel() for p in model2.parameters() if p.requires_grad)
total_b     = sum(p.numel() for p in model2.parameters())
print(f"Stage B trainable: {trainable_b:,} / {total_b:,}  ({100*trainable_b/total_b:.1f}%)")

opt2 = build_optimizer(model2, lr=5e-5, weight_decay=0.01)
sch2 = build_scheduler(opt2, warmup_steps=5, total_steps=50)

model2.train()
for step in range(3):
    batch = make_batch()
    opt2.zero_grad()
    out = model2(batch)
    out['loss'].backward()
    opt2.step(); sch2.step()
    print(f"  stage_b step {step+1}  loss={out['loss'].item():.4f}")

## 5. Checkpoint save / load

In [ ]:
import tempfile, os
from src.training.loop import save_checkpoint, load_checkpoint, TrainState

with tempfile.TemporaryDirectory() as tmpdir:
    ckpt_path = os.path.join(tmpdir, 'ckpt.pt')
    state = TrainState(step=5, epoch=1, best_val_loss=3.14)
    save_checkpoint(model2, opt2, sch2, state, ckpt_path)
    print(f"Checkpoint saved: {os.path.getsize(ckpt_path)/1024:.1f} KB")

    loaded_state = load_checkpoint(model2, opt2, sch2, ckpt_path, device='cpu')
    print(f"Loaded state: step={loaded_state.step}, epoch={loaded_state.epoch}, best_val={loaded_state.best_val_loss}")

---
**Summary**: Stage A freezes the decoder and trains only the projector. Stage B adds LoRA adapters to Q/V projections, enabling cheap decoder fine-tuning at ~1–2% trainable parameters.